In [1]:
from sklearn.svm import SVR
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn import datasets

In [2]:
df=datasets.load_diabetes(as_frame=True).frame

In [3]:
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [4]:
df.shape

(442, 11)

In [5]:
x=df.drop('target',axis=1)
y=df['target']

In [6]:
x.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [7]:
x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.3,random_state=42
)

In [8]:
y_scaler=StandardScaler()

y_train_scaled=y_scaler.fit_transform(y_train.values.reshape(-1,1)).ravel()
y_test_scaled=y_scaler.transform(y_test.values.reshape(-1,1)).ravel()

In [9]:
model=SVR()

model.fit(x_train,y_train_scaled)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [10]:
y_test_pred_scaled=model.predict(x_test)
y_train_pred_scaled=model.predict(x_train)

In [11]:
print("r2: ",r2_score(y_train_scaled,y_train_pred_scaled))
print("r2: ",r2_score(y_test_scaled,y_test_pred_scaled))

r2:  0.6596361676267712
r2:  0.48844443151651906


In [12]:
#linear
model=SVR(kernel='linear')

model.fit(x_train,y_train_scaled)
y_test_pred_scaled=model.predict(x_test)
y_train_pred_scaled=model.predict(x_train)

print("r2: ",r2_score(y_train_scaled,y_train_pred_scaled))
print("r2: ",r2_score(y_test_scaled,y_test_pred_scaled))

r2:  0.45191229982475245
r2:  0.4433761323833776


In [13]:
#Polynomial
model=SVR(kernel='poly')

model.fit(x_train,y_train_scaled)
y_test_pred_scaled=model.predict(x_test)
y_train_pred_scaled=model.predict(x_train)

print("r2: ",r2_score(y_train_scaled,y_train_pred_scaled))
print("r2: ",r2_score(y_test_scaled,y_test_pred_scaled))

r2:  0.5790920834310541
r2:  0.24203771038107758


In [14]:
#Sigmoid
model=SVR(kernel='sigmoid')
model.fit(x_train,y_train_scaled)

y_test_pred_scaled=model.predict(x_test)
y_train_pred_scaled=model.predict(x_train)

print("r2: ",r2_score(y_train_scaled,y_train_pred_scaled))
print("r2: ",r2_score(y_test_scaled,y_test_pred_scaled))

r2:  -19.721193440731305
r2:  -15.316808189576825


## Hyperparameter turing using gridsearch cv

In [15]:
from sklearn.model_selection import GridSearchCV

In [16]:
param_grid = {
    "C": [1, 2, 5, 10, 50, 100],
    "kernel": ["rbf", "linear"],
    "epsilon": [0.01, 0.1, 0.2, 0.3, 0.5]
}

In [18]:
svr = SVR()

grid_search = GridSearchCV(svr, param_grid, scoring="r2", cv=5)

grid_search.fit(x_train, y_train_scaled)

,estimator,SVR()
,param_grid,"{'C': [1, 2, ...], 'epsilon': [0.01, 0.1, ...], 'kernel': ['rbf', 'linear']}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,kernel,'linear'


In [19]:
print("best params - ", grid_search.best_params_)

best params -  {'C': 10, 'epsilon': 0.1, 'kernel': 'linear'}


In [20]:
best_model = SVR(kernel="linear", C=10, epsilon=0.1)

best_model.fit(x_train, y_train_scaled)

y_test_pred_scaled = best_model.predict(x_test)
y_train_pred_scaled = best_model.predict(x_train)

print("train r2: ", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2: ", r2_score(y_test_scaled, y_test_pred_scaled))

train r2:  0.5151066486918875
test r2:  0.47444183250401106


In [21]:
from sklearn.svm import LinearSVR

model = LinearSVR(C=10, epsilon=0.1, max_iter=5000)

model.fit(x_train, y_train_scaled)

y_test_pred_scaled = model.predict(x_test)
y_train_pred_scaled = model.predict(x_train)

print("train r2: ", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2: ", r2_score(y_test_scaled, y_test_pred_scaled))

train r2:  0.5152903661173718
test r2:  0.47427775373392544
